# Humanistica RDF4J MVP Notebook

Ce notebook produit une premiere analyse SPARQL-first directement sur le triple store RDF4J.

Sections:
1. Sanity check endpoint
2. Coverage KPI
3. Analyse temporelle
4. Analyse spatiale
5. Demonstration linked data (Wikidata federation)


In [ ]:
import os
import re
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)


In [ ]:
SPARQL_ENDPOINT = os.getenv('SPARQL_ENDPOINT')
SPARQL_USER = os.getenv('SPARQL_USER')
SPARQL_PASSWORD = os.getenv('SPARQL_PASSWORD')
SPARQL_TIMEOUT_SECONDS = int(os.getenv('SPARQL_TIMEOUT_SECONDS', '60'))
SPARQL_VERIFY_SSL = os.getenv('SPARQL_VERIFY_SSL', 'true').lower() == 'true'
USER_AGENT = os.getenv('USER_AGENT', 'Humanistica-Analytics/0.1')

missing = [k for k, v in {
    'SPARQL_ENDPOINT': SPARQL_ENDPOINT,
    'SPARQL_USER': SPARQL_USER,
    'SPARQL_PASSWORD': SPARQL_PASSWORD,
}.items() if not v]

if missing:
    raise RuntimeError(f"Variables d'environnement manquantes: {missing}")

print('Endpoint configure:', SPARQL_ENDPOINT)
print('SSL verify:', SPARQL_VERIFY_SSL)
print('Timeout (s):', SPARQL_TIMEOUT_SECONDS)


In [ ]:
def sparql_select(query: str, timeout: int | None = None) -> pd.DataFrame:
    headers = {
        'Accept': 'application/sparql-results+json',
        'User-Agent': USER_AGENT,
    }
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={'query': query},
        headers=headers,
        auth=(SPARQL_USER, SPARQL_PASSWORD),
        timeout=timeout or SPARQL_TIMEOUT_SECONDS,
        verify=SPARQL_VERIFY_SSL,
    )
    resp.raise_for_status()
    payload = resp.json()
    rows = payload.get('results', {}).get('bindings', [])

    out = []
    for row in rows:
        out.append({k: v.get('value') for k, v in row.items()})
    return pd.DataFrame(out)


def to_num(s):
    return pd.to_numeric(s, errors='coerce')


## 1) Sanity check endpoint

In [ ]:
q_graphs = '''
SELECT ?g (COUNT(?s) AS ?triples)
WHERE { GRAPH ?g { ?s ?p ?o } }
GROUP BY ?g
ORDER BY DESC(?triples)
'''

df_graphs = sparql_select(q_graphs)
df_graphs['triples'] = to_num(df_graphs['triples'])
df_graphs


In [ ]:
if df_graphs.empty:
    raise RuntimeError('Aucun graphe nomme trouve.')

DATA_GRAPH = df_graphs.iloc[0]['g']
MODEL_GRAPH = df_graphs.iloc[1]['g'] if len(df_graphs) > 1 else None

print('Data graph:', DATA_GRAPH)
print('Model graph:', MODEL_GRAPH)


In [ ]:
q_classes = f'''
SELECT ?class (COUNT(DISTINCT ?s) AS ?n)
WHERE {{ GRAPH <{DATA_GRAPH}> {{ ?s a ?class . }} }}
GROUP BY ?class
ORDER BY DESC(?n)
LIMIT 20
'''

df_classes = sparql_select(q_classes)
df_classes['n'] = to_num(df_classes['n'])
df_classes


## 2) Coverage KPI

In [ ]:
q_cov = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX supp: <https://sdhss.org/ontology/crm-supplement/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?metric ?total ?with_sameas
WHERE {{
  {{
    SELECT
      ('persons_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?person) AS ?total)
      (COUNT(DISTINCT ?personWith) AS ?with_sameas)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?person a crm:E21 ; supp:P20 ?eid .
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?person AS ?personWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('birth_commune_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?commune) AS ?total)
      (COUNT(DISTINCT ?communeWith) AS ?with_sameas)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?commune a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?commune), '/BirthCommune/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?commune AS ?communeWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('birth_departement_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?dep) AS ?total)
      (COUNT(DISTINCT ?depWith) AS ?with_sameas)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?dep a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?dep), '/BirthDepartement/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?dep AS ?depWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('death_commune_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?commune) AS ?total)
      (COUNT(DISTINCT ?communeWith) AS ?with_sameas)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?commune a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?commune), '/DeathCommune/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?commune AS ?communeWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('death_departement_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?dep) AS ?total)
      (COUNT(DISTINCT ?depWith) AS ?with_sameas)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?dep a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?dep), '/DeathDepartement/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?dep AS ?depWith)
        }}
      }}
    }}
  }}
}}
'''

df_cov = sparql_select(q_cov)
for c in ['total', 'with_sameas']:
    df_cov[c] = to_num(df_cov[c])
df_cov['coverage_pct'] = (100 * df_cov['with_sameas'] / df_cov['total']).round(2)
df_cov = df_cov.sort_values('coverage_pct', ascending=False)
df_cov


In [ ]:
fig_cov = px.bar(
    df_cov,
    x='metric',
    y='coverage_pct',
    text='coverage_pct',
    title='Couverture Wikidata (sameAs) par type d entite',
)
fig_cov.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_cov.update_layout(yaxis_title='Coverage (%)', xaxis_title='Metric')
fig_cov.show()


## 3) Analyse temporelle

In [ ]:
q_temporal = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?person ?name ?birth_date ?death_date
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    ?birth a crm:E67 ; crm:P98 ?person ; sdh:P1 ?birth_date .
    ?death a crm:E69 ; crm:P100 ?person ; sdh:P1 ?death_date .
    OPTIONAL {{ ?person sdh:P9 ?name . }}
  }}
}}
'''

df_temp = sparql_select(q_temporal)
df_temp['birth_dt'] = pd.to_datetime(df_temp['birth_date'], format='%d/%m/%Y', errors='coerce')
df_temp['death_dt'] = pd.to_datetime(df_temp['death_date'], format='%d/%m/%Y', errors='coerce')

df_temp['age_at_death'] = (df_temp['death_dt'] - df_temp['birth_dt']).dt.days / 365.25

df_temp['birth_decade'] = (df_temp['birth_dt'].dt.year // 10) * 10
df_temp['death_decade'] = (df_temp['death_dt'].dt.year // 10) * 10

n_total = len(df_temp)
valid_birth = df_temp['birth_dt'].notna().sum()
valid_death = df_temp['death_dt'].notna().sum()
valid_age = df_temp['age_at_death'].notna().sum()

print('Rows extracted:', n_total)
print('Valid birth dates:', valid_birth)
print('Valid death dates:', valid_death)
print('Valid age-at-death:', valid_age)
print('Dropped rows for age distribution:', n_total - valid_age)


In [ ]:
birth_series = (
    df_temp.dropna(subset=['birth_decade'])
    .groupby('birth_decade')
    .size()
    .reset_index(name='n_births')
)

death_series = (
    df_temp.dropna(subset=['death_decade'])
    .groupby('death_decade')
    .size()
    .reset_index(name='n_deaths')
)

fig_birth = px.line(
    birth_series,
    x='birth_decade',
    y='n_births',
    markers=True,
    title='Naissances par decennie'
)
fig_birth.update_layout(xaxis_title='Decennie', yaxis_title='Nombre')
fig_birth.show()

fig_death = px.line(
    death_series,
    x='death_decade',
    y='n_deaths',
    markers=True,
    title='Deces par decennie'
)
fig_death.update_layout(xaxis_title='Decennie', yaxis_title='Nombre')
fig_death.show()


In [ ]:
df_age = df_temp[df_temp['age_at_death'].between(10, 110, inclusive='both')].copy()

fig_age = px.histogram(
    df_age,
    x='age_at_death',
    nbins=40,
    title='Distribution de l age au deces (10-110 ans)',
)
fig_age.update_layout(xaxis_title='Age au deces', yaxis_title='Nombre de personnes')
fig_age.show()

print('Age median:', round(df_age['age_at_death'].median(), 2))
print('Age mean:', round(df_age['age_at_death'].mean(), 2))


## 4) Analyse spatiale

In [ ]:
q_spatial_top = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?type ?label (COUNT(DISTINCT ?person) AS ?n)
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    {{
      ?birth a crm:E67 ; crm:P98 ?person ; core:P6 ?place .
      FILTER(CONTAINS(STR(?place), '/BirthDepartement/'))
      ?place sdh:P9 ?label .
      BIND('BirthDepartement' AS ?type)
    }}
    UNION
    {{
      ?death a crm:E69 ; crm:P100 ?person ; core:P6 ?place .
      FILTER(CONTAINS(STR(?place), '/DeathDepartement/'))
      ?place sdh:P9 ?label .
      BIND('DeathDepartement' AS ?type)
    }}
  }}
}}
GROUP BY ?type ?label
ORDER BY DESC(?n)
'''

df_spatial_top = sparql_select(q_spatial_top)
df_spatial_top['n'] = to_num(df_spatial_top['n'])

df_birth_top = df_spatial_top[df_spatial_top['type'] == 'BirthDepartement'].nlargest(20, 'n')
df_death_top = df_spatial_top[df_spatial_top['type'] == 'DeathDepartement'].nlargest(20, 'n')

fig_birth_top = px.bar(df_birth_top.sort_values('n'), x='n', y='label', orientation='h', title='Top 20 departements de naissance')
fig_birth_top.update_layout(xaxis_title='Nombre de personnes', yaxis_title='Departement')
fig_birth_top.show()

fig_death_top = px.bar(df_death_top.sort_values('n'), x='n', y='label', orientation='h', title='Top 20 departements de deces')
fig_death_top.update_layout(xaxis_title='Nombre de personnes', yaxis_title='Departement')
fig_death_top.show()


In [ ]:
q_flows = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?birth_dep ?death_dep (COUNT(DISTINCT ?person) AS ?n)
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    ?birth a crm:E67 ; crm:P98 ?person ; core:P6 ?bDepNode .
    ?death a crm:E69 ; crm:P100 ?person ; core:P6 ?dDepNode .

    FILTER(CONTAINS(STR(?bDepNode), '/BirthDepartement/'))
    FILTER(CONTAINS(STR(?dDepNode), '/DeathDepartement/'))

    ?bDepNode sdh:P9 ?birth_dep .
    ?dDepNode sdh:P9 ?death_dep .
  }}
}}
GROUP BY ?birth_dep ?death_dep
ORDER BY DESC(?n)
'''

df_flows = sparql_select(q_flows)
df_flows['n'] = to_num(df_flows['n'])

print('Nombre de couples departement naissance->deces:', len(df_flows))
print('Top flux:')
df_flows.head(20)


In [ ]:
pivot = (
    df_flows
    .pivot_table(index='birth_dep', columns='death_dep', values='n', fill_value=0)
)

# Pour rester lisible, on limite aux departements les plus frequents en source/cible
row_strength = pivot.sum(axis=1).sort_values(ascending=False)
col_strength = pivot.sum(axis=0).sort_values(ascending=False)
keep_rows = row_strength.head(20).index
keep_cols = col_strength.head(20).index

heat = pivot.loc[keep_rows, keep_cols]

fig_heat = px.imshow(
    heat,
    labels={'x': 'Departement de deces', 'y': 'Departement de naissance', 'color': 'Nombre'},
    title='Matrice des flux naissance -> deces (Top 20 x Top 20)',
    aspect='auto',
)
fig_heat.show()


## 5) Demonstration linked data (Wikidata federation)

In [ ]:
q_map_template = '''
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX supp: <https://sdhss.org/ontology/crm-supplement/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?commune ?name ?wd ?coord
WHERE {{{
  {{
    SELECT ?commune ?name ?wd
    WHERE {{{{
      GRAPH <{DATA_GRAPH}> {{{
        ?commune a core:C13 ; supp:P20 ?eid ; sdh:P9 ?name .
        FILTER(CONTAINS(STR(?commune), '/BirthCommune/'))
        ?eid owl:sameAs ?wd .
        FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
      }}
    }}
    LIMIT {limit}
  }}
  SERVICE <https://query.wikidata.org/sparql> {{
    ?wd wdt:P625 ?coord .
  }}
}}
'''

# Progressive strategy to avoid federation timeouts
attempt_limits = [50, 100, 150, 250]
df_map = pd.DataFrame()
last_error = None

for lim in attempt_limits:
    try:
        q_map = q_map_template.format(limit=lim, DATA_GRAPH=DATA_GRAPH)
        candidate = sparql_select(q_map, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
        if not candidate.empty and 'coord' in candidate.columns:
            df_map = candidate.copy()
            print(f'Federation OK with LIMIT={lim}: {len(df_map)} rows')
            break
        print(f'Federation returned no usable rows with LIMIT={lim}')
    except Exception as e:
        last_error = e
        print(f'Federation failed with LIMIT={lim}: {type(e).__name__}: {e}')

if df_map.empty:
    print('Federation did not return usable coordinate rows.')
    if last_error is not None:
        print('Last error:', type(last_error).__name__, str(last_error))
else:
    display(df_map.head())


In [ ]:
def parse_wkt_point(wkt: str):
    m = re.match(r'^Point\(([-0-9.]+) ([-0-9.]+)\)$', str(wkt))
    if not m:
        return (np.nan, np.nan)
    lon, lat = float(m.group(1)), float(m.group(2))
    return lon, lat

if 'df_map' not in globals() or df_map.empty or 'coord' not in df_map.columns:
    print('No coordinate payload available. Skipping geo parsing.')
    df_geo = pd.DataFrame(columns=['name', 'wd', 'lat', 'lon'])
else:
    coords = df_map['coord'].apply(parse_wkt_point)
    df_map['lon'] = coords.apply(lambda x: x[0])
    df_map['lat'] = coords.apply(lambda x: x[1])

    df_geo = df_map.dropna(subset=['lat', 'lon']).copy()
    print('Rows with valid coordinates:', len(df_geo))


In [ ]:
if df_geo.empty:
    print('No map to display: dataframe is empty after federation/parsing.')
else:
    fig_geo = px.scatter_geo(
        df_geo,
        lat='lat',
        lon='lon',
        hover_name='name',
        hover_data={'wd': True, 'lat': ':.4f', 'lon': ':.4f'},
        title='Communes de naissance enrichies via Wikidata (sample)',
        opacity=0.7,
    )
    fig_geo.update_layout(geo_scope='europe')
    fig_geo.show()


## Conclusions rapides

- Le pipeline SPARQL-first fonctionne sur RDF4J avec authentification.
- Les analyses coverage, temporelles et spatiales sont produites sans CSV source.
- La federation Wikidata fonctionne, mais doit etre limitee (sous-requete `LIMIT`) pour eviter les timeouts.
- Prochaine etape: version narrative propre (moins de cellules techniques, plus de messages historiques).